# 01 — Extract FashionCLIP image embeddings

One-time pass over **items_train** + **items_phase_1** (and items_phase_2 once released)
using `patrickjohncyh/fashion-clip`. Writes a `dict[str, torch.Tensor (512,)]` to
`phase2/embeddings/fashion_clip_image.pt`.

Domain-tuned image embeddings should help the model distinguish visually similar
fashion items (different white shoes, different black tees) where vanilla CLIP saturates.

**Cost:** ~few hours on M4 MPS at batch 64. Run once.


In [ ]:
import sys, os
sys.path.insert(0, '/Users/matouskovar/FIT/adm-sp/src')

REPO_ROOT     = '/Users/matouskovar/FIT/adm-sp'
DATA_DIR      = f'{REPO_ROOT}/data'
ARTIFACTS_DIR = f'{REPO_ROOT}/artifacts'
PHASE2_DIR    = f'{REPO_ROOT}/phase2'

# Read-only artifacts
TEXT_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/text_multilingual.pt'
CLIP_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/clip_image.pt'   # baseline (vanilla CLIP)
PIPELINE_PATH      = f'{ARTIFACTS_DIR}/pipelines/preprocessing.pkl'
VOCAB_DIR          = f'{ARTIFACTS_DIR}/vocabularies'
BASELINE_MODEL     = f'{ARTIFACTS_DIR}/models/siamese_baseline.pth'   # F1 ~ 0.85 on val
HARDMINED_MODEL    = f'{ARTIFACTS_DIR}/models/siamese_hardmined.pth'  # F1 ~ 0.0004 (broken)

# Phase 2 outputs
FASHION_EMB_PATH   = f'{PHASE2_DIR}/embeddings/fashion_clip_image.pt'
LEARNED_EMB_PATH   = f'{PHASE2_DIR}/embeddings/learned_embeddings.pt'
SIAMESE_V2_PATH    = f'{PHASE2_DIR}/models/siamese_v2.pth'
METRIC_V2_PATH     = f'{PHASE2_DIR}/models/metric_v2.pth'

import torch
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# Dependencies (run once)
# %pip install transformers pillow tqdm pandas torch


In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor

IMAGES_DIR = '/Users/matouskovar/FIT/images'   # unchanged from baseline
BATCH_SIZE = 64

# Collect all itemIds we need to embed (train + phase1 + phase2 if present)
df_train = pd.read_csv(f'{DATA_DIR}/items_train.csv', usecols=['itemId'])
df_p1    = pd.read_csv(f'{DATA_DIR}/items_phase_1.csv', usecols=['itemId'])
all_ids  = set(df_train['itemId'].astype(str)) | set(df_p1['itemId'].astype(str))

p2_path = f'{DATA_DIR}/items_phase_2.csv'
if os.path.exists(p2_path):
    df_p2 = pd.read_csv(p2_path, usecols=['itemId'])
    all_ids |= set(df_p2['itemId'].astype(str))

print(f'Total items to embed: {len(all_ids):,}')


In [ ]:
model_id = 'patrickjohncyh/fashion-clip'
print(f'Loading {model_id}...')
clip = CLIPModel.from_pretrained(model_id).to(device).eval()
proc = CLIPProcessor.from_pretrained(model_id)
print(f'  Image embedding dim: {clip.config.projection_dim}')   # expected 512


In [ ]:
@torch.no_grad()
def embed_batch(item_ids):
    imgs, kept_ids = [], []
    for iid in item_ids:
        try:
            img = Image.open(f'{IMAGES_DIR}/{iid}.jpg').convert('RGB')
            imgs.append(img); kept_ids.append(iid)
        except Exception:
            pass   # missing/corrupt image -> skip; will be a singleton at clustering
    if not imgs:
        return {}
    inputs = proc(images=imgs, return_tensors='pt').to(device)
    feats  = clip.get_image_features(**inputs)
    feats  = feats.cpu()
    return {iid: feats[i] for i, iid in enumerate(kept_ids)}

# Resume-safe: load partial output if it exists
out_path = FASHION_EMB_PATH
os.makedirs(os.path.dirname(out_path), exist_ok=True)

if os.path.exists(out_path):
    embs = torch.load(out_path, map_location='cpu', weights_only=False)
    print(f'Resuming with {len(embs):,} already-embedded items')
else:
    embs = {}

remaining = sorted(all_ids - set(embs.keys()))
print(f'Remaining to embed: {len(remaining):,}')


In [ ]:
CHECKPOINT_EVERY = 50_000   # save partial every N items

processed_since_save = 0
for start in tqdm(range(0, len(remaining), BATCH_SIZE)):
    batch_ids = remaining[start : start + BATCH_SIZE]
    embs.update(embed_batch(batch_ids))
    processed_since_save += len(batch_ids)
    if processed_since_save >= CHECKPOINT_EVERY:
        torch.save(embs, out_path)
        processed_since_save = 0

torch.save(embs, out_path)
print(f'\nSaved {len(embs):,} embeddings to {out_path}')


## Sanity check — same vs different product

Verify FashionCLIP embeddings cluster items with the same `label` more tightly than
items in different labels (in `items_train`). If FashionCLIP doesn't beat vanilla CLIP
here, no point continuing with Approach 1.


In [ ]:
import torch.nn.functional as F
import random
random.seed(0)

df_train_full = pd.read_csv(f'{DATA_DIR}/items_train.csv', usecols=['itemId','label'])
df_train_full['itemId'] = df_train_full['itemId'].astype(str)
df_train_full = df_train_full[df_train_full['itemId'].isin(embs)]

# Sample 1000 same-label pairs and 1000 different-label pairs
groups = df_train_full.groupby('label')
multi  = [g for _, g in groups if len(g) >= 2]
random.shuffle(multi)

same_pairs, diff_pairs = [], []
for g in multi[:2000]:
    a, b = g['itemId'].sample(2, random_state=len(same_pairs)).tolist()
    same_pairs.append((a, b))
    if len(same_pairs) >= 1000: break

ids_pool = df_train_full['itemId'].tolist()
while len(diff_pairs) < 1000:
    a, b = random.sample(ids_pool, 2)
    if df_train_full.set_index('itemId').loc[a,'label'] != df_train_full.set_index('itemId').loc[b,'label']:
        diff_pairs.append((a, b))

def cos(a, b):
    return F.cosine_similarity(embs[a].unsqueeze(0), embs[b].unsqueeze(0)).item()

same_sims = [cos(a,b) for a,b in same_pairs]
diff_sims = [cos(a,b) for a,b in diff_pairs]
print(f'Same-label cos sim: mean={sum(same_sims)/len(same_sims):.3f}, min={min(same_sims):.3f}')
print(f'Diff-label cos sim: mean={sum(diff_sims)/len(diff_sims):.3f}, max={max(diff_sims):.3f}')
print(f'Gap: {sum(same_sims)/len(same_sims) - sum(diff_sims)/len(diff_sims):.3f}')
